# Azure AI Foundry Launcher — Healthcare Demo

Run cells in order to automate the complete Foundry IQ setup for the Healthcare Orchestrator Agent.

---

## Where to Run This Notebook

**This is not a Fabric notebook.** Unlike `Healthcare_Launcher.ipynb` (which runs inside Fabric), this notebook needs Azure CLI access and runs outside of Fabric. The three best options:

| Option | How | Notes |
|--------|-----|-------|
| **This Codespace (recommended)** | Open this file in VS Code, select a Python kernel, run cells | Azure CLI is pre-installed; run `az login` in the terminal first |
| **Azure Cloud Shell** | portal.azure.com → Cloud Shell → upload this file, open with Jupyter | Az CLI pre-authenticated; no local install needed |
| **Local VS Code** | Python 3.9+, `az login` completed | Works fine; same experience as Codespace |

> **Default region in this notebook is now Sweden Central (`swedencentral`)**. Keep your Fabric workspace in the same region.
> **Before running Cell 3**: open the VS Code terminal and run `az login` to authenticate the CLI. The notebook will auto-detect your subscription.

---

## ⚠️ Before You Run — One Manual Prerequisite

> **Activate your Global Admin role via PIM first.**
> Azure Portal → Microsoft Entra ID → Privileged Identity Management → My roles → Activate *Global Administrator*.
> This is an interactive MFA step that cannot be scripted.

---

## Execution Dependencies (Critical)

Foundry configuration must not start until:

1. Critical RBAC is complete and propagated.
2. `Healthcare_Launcher.ipynb` has completed in Fabric and produced required artifacts.

This notebook now enforces those dependencies with a hard gate before any Foundry configuration calls.

---

## What Gets Deployed

| Cell | Step | Action | Method |
|------|------|--------|--------|
| 4 | 0 | Enable Fabric Tenant Settings | Power Platform Admin REST API |
| 5 | 1 | Create Foundry Hub + Project | Azure CLI + ARM REST |
| 6 | 2 | Deploy gpt-4o + embedding models | Azure CLI |
| 7 | 3 | Create Azure AI Search (Basic, MSI enabled) | Azure CLI |
| 8 | 4 | Assign critical RBAC on Search + AI Services + Fabric workspace, then wait for propagation | Azure CLI + Fabric REST API |
| 9 | 5 | Validate Fabric prerequisites from `Healthcare_Launcher.ipynb` (depends-on gate) | Fabric REST API |
| 10 | 6 | Connect Search → Foundry | Foundry REST API |
| 11 | 7 | Create Knowledge Source (OneLake) | Foundry REST API *(preview — may need manual fallback)* |
| 12 | 8 | Create Knowledge Base | Foundry REST API *(preview — may need manual fallback)* |
| 13 | 9 | Poll indexer until success | AI Search REST API |
| 14 | 10 | Connect Fabric Data Agent → Foundry | Foundry REST API *(may need manual fallback — see below)* |
| 15 | 11 | Create Orchestrator Agent (gpt-4o) | Foundry REST API |
| 16 | 12 | Run 3 validation queries | Foundry REST API |

---

## What Still Requires Manual Steps After Running

Three steps use undocumented preview APIs or portal-specific behavior. Each cell prints detailed instructions if it falls back, but here is the consolidated list:

### 🔴 Cell 14 — Fabric Data Agent Connection (most likely to need manual)
The Foundry portal stores internal workspace/artifact routing when you pick a Data Agent that isn't exposed in the REST API. If Cell 14's automated attempt doesn't wire up correctly, the agent tool returns *"No tool output found"* silently.

**Manual fix:** `ai.azure.com → Management center → Connected resources → + New connection → Microsoft Fabric → Data Agent`
Provide Workspace ID and Artifact ID from Cell 2 config, name it `HealthcareHLSAgent`.

### 🟡 Cells 11–12 — Knowledge Source + Knowledge Base (may need manual)
These use undocumented Foundry IQ preview endpoints that may return 404/405.

**Manual fix:** `ai.azure.com → Knowledge → + Add knowledge source → OneLake`
Then `Knowledge bases → + Create`, point it at the new source.

### 🟡 Cell 15 — `KNOWLEDGE_INDEX_NAME` variable (only if Cell 11 fell back)
If Cell 11 ran manually, the auto-generated index name (e.g. `healthcare-onelake-ks-index`) won't be captured. Look it up in **Azure Portal → AI Search → Indexes** and set it in Cell 15 before running:
```python
KNOWLEDGE_INDEX_NAME = "your-actual-index-name"
```

---

## Prerequisites
- Azure CLI installed and logged in (`az login`) — see "Where to Run" above
- Python 3.9+
- Fabric Healthcare Demo already deployed (`Healthcare_Launcher.ipynb` completed)
- PIM Global Admin role activated (see above)

In [ ]:
# ============================================================================
# CELL 1 — Install dependencies
# ============================================================================

%pip install azure-identity azure-mgmt-cognitiveservices azure-mgmt-search \
    azure-mgmt-authorization azure-mgmt-resource azure-ai-projects \
    requests --quiet --disable-pip-version-check

print("Dependencies installed")

In [ ]:
# ============================================================================
# CELL 2 — CONFIGURATION  ← Edit these values before running
# ============================================================================

# --- Azure subscription / resource group ---
SUBSCRIPTION_ID    = ""                          # az account show --query id -o tsv
RESOURCE_GROUP     = "healthcaredemofoundry-rg"  # Will be created if it doesn't exist
LOCATION           = "swedencentral"            # Default region. Keep this aligned with your Fabric workspace region.

# --- Azure AI Foundry Hub + Project ---
HUB_NAME           = "HLS-HealthcareDemo"        # Becomes the AI Services account name
PROJECT_NAME       = "HealthcareDemo-HLS"        # Foundry project name

# --- Models ---
CHAT_MODEL                  = "gpt-4o"
CHAT_MODEL_VERSION          = "2024-11-20"       # Use latest available in your region
CHAT_MODEL_TPM              = 80                 # Thousands of tokens per minute
EMBEDDING_MODEL             = "text-embedding-ada-002"
EMBEDDING_MODEL_VERSION     = "2"
EMBEDDING_MODEL_TPM         = 120                # Thousands of tokens per minute

# --- Azure AI Search ---
SEARCH_SERVICE_NAME         = "healthcarefoundryais"  # Must be globally unique
SEARCH_SKU                  = "basic"                 # minimum for managed identity

# --- Fabric (from Healthcare_Launcher output) ---
FABRIC_WORKSPACE_ID         = ""   # Fabric portal → Workspace → Settings → ID
FABRIC_LAKEHOUSE_ID         = ""   # Fabric portal → lh_bronze_raw → URL → after /lakehouses/
DATA_AGENT_ARTIFACT_ID      = ""   # Fabric portal → HealthcareHLSAgent → URL → after /aiskills/
KNOWLEDGE_FOLDER_PATH       = "Files/healthcare_knowledge"   # path inside lakehouse Files

# --- Foundry resource names (safe to leave as defaults) ---
SEARCH_CONNECTION_NAME      = "healthcareaisearch"
DATA_AGENT_CONNECTION_NAME  = "HealthcareHLSAgent"
KNOWLEDGE_SOURCE_NAME       = "healthcare-onelake-ks"
KNOWLEDGE_BASE_NAME         = "healthcareknowledgebase"
ORCHESTRATOR_AGENT_NAME     = "HealthcareOrchestratorAgent"

# ---- Validation ----
assert SUBSCRIPTION_ID,    "Set SUBSCRIPTION_ID"
assert FABRIC_WORKSPACE_ID, "Set FABRIC_WORKSPACE_ID"
assert FABRIC_LAKEHOUSE_ID, "Set FABRIC_LAKEHOUSE_ID"
assert DATA_AGENT_ARTIFACT_ID, "Set DATA_AGENT_ARTIFACT_ID"

FOUNDRY_ENDPOINT = f"https://{HUB_NAME}.services.ai.azure.com"
FOUNDRY_API_BASE = f"{FOUNDRY_ENDPOINT}/api/projects/{PROJECT_NAME}"

print("Configuration valid")
print(f"  Hub:      {HUB_NAME}")
print(f"  Project:  {PROJECT_NAME}")
print(f"  Region:   {LOCATION}")
print(f"  Foundry:  {FOUNDRY_API_BASE}")

In [ ]:
# ============================================================================
# CELL 3 — Imports + helper utilities
# ============================================================================

import json
import subprocess
import time
import sys
import requests
from pathlib import Path


def az(cmd: str, *, json_output: bool = True, check: bool = True):
    """Run an az CLI command. Returns parsed JSON or raw stdout."""
    full_cmd = f"az {cmd}" + (" -o json" if json_output else "")
    result = subprocess.run(full_cmd, shell=True, capture_output=True, text=True)
    if check and result.returncode != 0:
        err = result.stderr.strip()
        # 'already exists' errors are OK — treat as idempotent
        if "already exists" in err or "AlreadyExists" in err or "Conflict" in err:
            print(f"  [exists] {err[:120]}")
            return None
        raise RuntimeError(f"az command failed:\n{err}")
    if json_output and result.stdout.strip():
        try:
            return json.loads(result.stdout)
        except json.JSONDecodeError:
            return result.stdout.strip()
    return result.stdout.strip()


def get_token(resource: str) -> str:
    """Get an Azure access token for the given resource audience."""
    result = subprocess.run(
        ["az", "account", "get-access-token", "--resource", resource, "--query", "accessToken", "-o", "tsv"],
        capture_output=True, text=True, check=True
    )
    return result.stdout.strip()


def foundry_request(method: str, path: str, body: dict = None, api_version: str = "2025-05-15-preview"):
    """Call the Azure AI Foundry project REST API."""
    token = get_token("https://ai.azure.com")
    url = f"{FOUNDRY_API_BASE}/{path.lstrip('/')}?api-version={api_version}"
    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
    resp = requests.request(method, url, headers=headers, json=body, timeout=60)
    if resp.status_code in (200, 201, 202):
        return resp.json() if resp.content else {}
    if resp.status_code == 409:
        print(f"  [exists] {path}")
        return None
    resp.raise_for_status()


def fabric_request(method: str, path: str, body: dict = None):
    """Call the Microsoft Fabric REST API."""
    token = get_token("https://api.fabric.microsoft.com")
    url = f"https://api.fabric.microsoft.com/v1/{path.lstrip('/')}"
    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
    resp = requests.request(method, url, headers=headers, json=body, timeout=60)
    if resp.status_code in (200, 201, 202):
        return resp.json() if resp.content else {}
    if resp.status_code == 409:
        print(f"  [exists] {path}")
        return None
    resp.raise_for_status()


def arm_request(method: str, path: str, body: dict = None, api_version: str = "2024-10-01"):
    """Call the Azure Resource Manager REST API."""
    token = get_token("https://management.azure.com")
    url = f"https://management.azure.com/{path.lstrip('/')}?api-version={api_version}"
    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
    resp = requests.request(method, url, headers=headers, json=body, timeout=120)
    if resp.status_code in (200, 201, 202):
        return resp.json() if resp.content else {}
    if resp.status_code == 409:
        print(f"  [exists] {path}")
        return None
    resp.raise_for_status()


# Quick az-login check
acct = az("account show")
print(f"Logged in as: {acct['user']['name']}")
print(f"Subscription: {acct['name']} ({acct['id']})")
if not SUBSCRIPTION_ID:
    SUBSCRIPTION_ID = acct["id"]
    print(f"  Auto-detected SUBSCRIPTION_ID: {SUBSCRIPTION_ID}")

# Confirm correct subscription
az(f"account set --subscription {SUBSCRIPTION_ID}", json_output=False)
USER_OBJECT_ID = az("ad signed-in-user show --query id -o tsv", json_output=False)
print(f"User Object ID: {USER_OBJECT_ID}")

In [ ]:
# ============================================================================
# CELL 4 — [Step 0] Fabric Tenant Settings
# Requires Fabric Tenant Admin role. If you are not a tenant admin, the API
# calls will return 403 — follow the manual instructions printed below.
# ============================================================================

TENANT_SETTINGS = [
    {"settingName": "ServicePrincipalAccess",           "label": "Service Principals can use Fabric APIs"},
    {"settingName": "UsersCanCreateFabricItems",         "label": "Users can create Fabric items (Graph/Ontology)"},
    {"settingName": "AISkillTenantSwitch",               "label": "Users can create Data Agents (AISkill)"},
    {"settingName": "OperationsAgentTenantSwitch",        "label": "Enable Operations Agent"},
]

print("=" * 60)
print("STEP 0 — Fabric Tenant Settings")
print("=" * 60)

try:
    fabric_token = get_token("https://api.fabric.microsoft.com")
    current = requests.get(
        "https://api.fabric.microsoft.com/v1/admin/tenantsettings",
        headers={"Authorization": f"Bearer {fabric_token}"},
        timeout=30
    )
    if current.status_code == 403:
        raise PermissionError("403 — Not a Fabric Tenant Admin")
    current.raise_for_status()
    existing = {s["settingName"]: s for s in current.json().get("tenantSettings", [])}

    for setting in TENANT_SETTINGS:
        name = setting["settingName"]
        label = setting["label"]
        current_state = existing.get(name, {})
        if current_state.get("enabled"):
            print(f"  ✅ Already enabled: {label}")
            continue
        # Enable the setting
        patch_resp = requests.patch(
            "https://api.fabric.microsoft.com/v1/admin/tenantsettings",
            headers={"Authorization": f"Bearer {fabric_token}", "Content-Type": "application/json"},
            json={"settingName": name, "enabled": True},
            timeout=30
        )
        if patch_resp.status_code in (200, 204):
            print(f"  ✅ Enabled: {label}")
        else:
            print(f"  ⚠️  Could not enable {label}: {patch_resp.status_code} {patch_resp.text[:80]}")

except PermissionError as e:
    print(f"\n  ⚠️  {e}")
    print("  Manual steps required in the Fabric Admin portal:")
    print("  1. Go to app.fabric.microsoft.com → Settings (gear) → Admin portal")
    print("  2. Tenant settings → Developer settings → Enable 'Service Principals can use Fabric APIs'")
    print("  3. Tenant settings → Enable 'Users can create Graph'")
    print("  4. Tenant settings → Enable 'Users can create Ontology'")
    print("  5. Tenant settings → Preview features → Enable 'Operations Agent'")
    print("\n  Continue to Cell 5 after completing the above.")

In [ ]:
# ============================================================================
# CELL 5 — [Step 1] Register providers, create resource group + Foundry Hub
# ============================================================================

print("=" * 60)
print("STEP 1 — Create Foundry Hub + Project")
print("=" * 60)

# Register required providers (idempotent)
print("Registering Azure providers...")
for ns in ["Microsoft.CognitiveServices", "Microsoft.Search", "Microsoft.MachineLearningServices"]:
    az(f"provider register --namespace {ns}", json_output=False, check=False)
print("  Providers registered (propagation may take a few minutes)")

# Create resource group
print(f"Creating resource group '{RESOURCE_GROUP}' in {LOCATION}...")
rg = az(f"group create --name {RESOURCE_GROUP} --location {LOCATION}")
print(f"  Resource group: {rg['properties']['provisioningState'] if rg else 'exists'}")

# Create AI Services account (this IS the Foundry Hub)
print(f"Creating AI Services account '{HUB_NAME}' (Foundry Hub)...")
hub = az(
    f"cognitiveservices account create"
    f" --name {HUB_NAME}"
    f" --resource-group {RESOURCE_GROUP}"
    f" --kind AIServices"
    f" --sku S0"
    f" --location {LOCATION}"
    f" --assign-identity"
    f" --yes"
)
if hub is None:
    # Already exists — fetch it
    hub = az(f"cognitiveservices account show --name {HUB_NAME} --resource-group {RESOURCE_GROUP}")

HUB_RESOURCE_ID = hub["id"]
FOUNDRY_HUB_MSI = hub["identity"]["principalId"]
print(f"  Hub resource ID: {HUB_RESOURCE_ID}")
print(f"  Hub MSI (principal): {FOUNDRY_HUB_MSI}")

# Create Foundry Project (sub-resource of the AI Services account)
print(f"Creating Foundry project '{PROJECT_NAME}'...")
project_path = (
    f"subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP}"
    f"/providers/Microsoft.CognitiveServices/accounts/{HUB_NAME}/projects/{PROJECT_NAME}"
)
project = arm_request("PUT", project_path, body={
    "location": LOCATION,
    "identity": {"type": "SystemAssigned"},
    "properties": {"displayName": PROJECT_NAME, "description": "Healthcare Demo project"}
})
if project is None:
    project = arm_request("GET", project_path)

PROJECT_MSI = project.get("identity", {}).get("principalId", "")
print(f"  Project MSI: {PROJECT_MSI}")
print(f"  ✅ Foundry Hub + Project ready")
print(f"  Foundry endpoint: {FOUNDRY_ENDPOINT}")

In [ ]:
# ============================================================================
# CELL 6 — [Step 2] Deploy gpt-4o + text-embedding-ada-002
# ============================================================================

print("=" * 60)
print("STEP 2 — Deploy Models")
print("=" * 60)

MODELS_TO_DEPLOY = [
    {
        "name": CHAT_MODEL,
        "model_format": "OpenAI",
        "model_name": CHAT_MODEL,
        "model_version": CHAT_MODEL_VERSION,
        "capacity": CHAT_MODEL_TPM,
        "label": f"Chat model ({CHAT_MODEL})",
    },
    {
        "name": EMBEDDING_MODEL,
        "model_format": "OpenAI",
        "model_name": EMBEDDING_MODEL,
        "model_version": EMBEDDING_MODEL_VERSION,
        "capacity": EMBEDDING_MODEL_TPM,
        "label": f"Embedding model ({EMBEDDING_MODEL})",
    },
]

for m in MODELS_TO_DEPLOY:
    print(f"Deploying {m['label']}...")
    result = az(
        f"cognitiveservices account deployment create"
        f" --name {HUB_NAME}"
        f" --resource-group {RESOURCE_GROUP}"
        f" --deployment-name {m['name']}"
        f" --model-name {m['model_name']}"
        f" --model-version {m['model_version']}"
        f" --model-format {m['model_format']}"
        f" --sku-name Standard"
        f" --sku-capacity {m['capacity']}"
    )
    if result:
        state = result.get("properties", {}).get("provisioningState", "unknown")
        print(f"  ✅ {m['label']}: {state} — {m['capacity']}K TPM")
    else:
        print(f"  ✅ {m['label']}: already exists")

print("Models deployed")

In [ ]:
# ============================================================================
# CELL 7 — [Step 3] Create Azure AI Search (Basic, MSI on, auth = Both)
# ============================================================================

print("=" * 60)
print("STEP 3 — Create Azure AI Search")
print("=" * 60)

print(f"Creating search service '{SEARCH_SERVICE_NAME}' (sku={SEARCH_SKU})...")
search = az(
    f"search service create"
    f" --name {SEARCH_SERVICE_NAME}"
    f" --resource-group {RESOURCE_GROUP}"
    f" --sku {SEARCH_SKU}"
    f" --location {LOCATION}"
    f" --identity-type SystemAssigned"
    f" --auth-options aadOrApiKey"
)
if search is None:
    search = az(f"search service show --name {SEARCH_SERVICE_NAME} --resource-group {RESOURCE_GROUP}")

SEARCH_RESOURCE_ID = search["id"]
SEARCH_MSI = search["identity"]["principalId"]
SEARCH_ENDPOINT = f"https://{SEARCH_SERVICE_NAME}.search.windows.net"

# Retrieve the admin API key (needed for indexer verification in Cell 11)
search_keys = az(f"search admin-key show --service-name {SEARCH_SERVICE_NAME} --resource-group {RESOURCE_GROUP}")
SEARCH_ADMIN_KEY = search_keys["primaryKey"]

print(f"  Search endpoint:  {SEARCH_ENDPOINT}")
print(f"  Search MSI:       {SEARCH_MSI}")
print(f"  ✅ Azure AI Search ready")

In [ ]:
# ============================================================================
# CELL 8 — [Step 4] Assign 7 RBAC roles
# Role assignments are idempotent — duplicates are silently skipped.
# ============================================================================

print("=" * 60)
print("STEP 4 — Assign RBAC Roles")
print("=" * 60)

AI_SERVICES_SCOPE = (
    f"/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP}"
    f"/providers/Microsoft.CognitiveServices/accounts/{HUB_NAME}"
)
SEARCH_SCOPE = (
    f"/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP}"
    f"/providers/Microsoft.Search/searchServices/{SEARCH_SERVICE_NAME}"
)

ROLE_ASSIGNMENTS = [
    # --- Your user on AI Search (3 roles) ---
    (USER_OBJECT_ID,  "Search Index Data Contributor", SEARCH_SCOPE,      "user → Search: Index Data Contributor"),
    (USER_OBJECT_ID,  "Search Index Data Reader",      SEARCH_SCOPE,      "user → Search: Index Data Reader"),
    (USER_OBJECT_ID,  "Search Service Contributor",    SEARCH_SCOPE,      "user → Search: Service Contributor"),
    # --- Search MSI on AI Search (same 3 roles) ---
    (SEARCH_MSI,      "Search Index Data Contributor", SEARCH_SCOPE,      "Search MSI → Search: Index Data Contributor"),
    (SEARCH_MSI,      "Search Index Data Reader",      SEARCH_SCOPE,      "Search MSI → Search: Index Data Reader"),
    (SEARCH_MSI,      "Search Service Contributor",    SEARCH_SCOPE,      "Search MSI → Search: Service Contributor"),
    # --- Search MSI on AI Services (for embedding model calls) ---
    (SEARCH_MSI,      "Cognitive Services OpenAI User",        AI_SERVICES_SCOPE, "Search MSI → AI Services: OpenAI User"),
    (SEARCH_MSI,      "Cognitive Services OpenAI Contributor", AI_SERVICES_SCOPE, "Search MSI → AI Services: OpenAI Contributor"),
]

for assignee, role, scope, label in ROLE_ASSIGNMENTS:
    result = az(
        f'role assignment create --assignee "{assignee}" --role "{role}" --scope "{scope}"',
        check=False
    )
    if result:
        print(f"  ✅ {label}")
    else:
        print(f"  ✅ {label} [already assigned]")

print("\nAll RBAC roles assigned")
print("Waiting 10 minutes for role propagation...")
for i in range(10, 0, -1):
    print(f"  {i} min remaining...", end="\r")
    time.sleep(60)
print("Role propagation wait complete            ")

In [ ]:
# ============================================================================
# CELL 9 — [Step 5] Connect Azure AI Search to Foundry project
# Uses the Foundry Connections REST API (preview).
# ============================================================================

print("=" * 60)
print("STEP 5 — Connect Search → Foundry")
print("=" * 60)

search_connection_body = {
    "name": SEARCH_CONNECTION_NAME,
    "type": "CognitiveSearch",
    "target": SEARCH_ENDPOINT,
    "authType": "AAD",
    "metadata": {
        "ApiType": "Azure",
        "ResourceId": SEARCH_RESOURCE_ID,
        "location": LOCATION,
    }
}

conn = foundry_request("PUT", f"connections/{SEARCH_CONNECTION_NAME}", search_connection_body)
if conn:
    SEARCH_CONNECTION_ID = conn.get("id", f"{FOUNDRY_API_BASE}/connections/{SEARCH_CONNECTION_NAME}")
    print(f"  ✅ Search connection created: {SEARCH_CONNECTION_NAME}")
else:
    # Fetch existing
    conn = foundry_request("GET", f"connections/{SEARCH_CONNECTION_NAME}")
    SEARCH_CONNECTION_ID = conn.get("id", f"{FOUNDRY_API_BASE}/connections/{SEARCH_CONNECTION_NAME}")
    print(f"  ✅ Search connection exists: {SEARCH_CONNECTION_NAME}")

# Build ARM-style connection resource ID for use in agent tools
SEARCH_CONNECTION_ARM_ID = (
    f"/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP}"
    f"/providers/Microsoft.CognitiveServices/accounts/{HUB_NAME}"
    f"/projects/{PROJECT_NAME}/connections/{SEARCH_CONNECTION_NAME}"
)
print(f"  Connection ARM ID: {SEARCH_CONNECTION_ARM_ID}")

In [ ]:
# ============================================================================
# CELL 10 — [Steps 10b] Grant MSI access to Fabric Workspace
# Both the Foundry Hub MSI and Search MSI need Contributor on the workspace
# so the indexer can read files from OneLake.
# ============================================================================

print("=" * 60)
print("STEP 10b — Grant MSI Access to Fabric Workspace")
print("=" * 60)

FABRIC_MSI_GRANTS = [
    (FOUNDRY_HUB_MSI, "Contributor", f"Foundry Hub MSI ({HUB_NAME})"),
    (SEARCH_MSI,      "Contributor", f"Search MSI ({SEARCH_SERVICE_NAME})"),
]

for principal_id, role, label in FABRIC_MSI_GRANTS:
    try:
        result = fabric_request(
            "POST",
            f"workspaces/{FABRIC_WORKSPACE_ID}/roleAssignments",
            body={
                "principal": {"id": principal_id, "type": "ServicePrincipal"},
                "role": role,
            }
        )
        print(f"  ✅ {label} → Fabric workspace: {role}")
    except requests.HTTPError as e:
        if e.response.status_code == 409:
            print(f"  ✅ {label} [already has {role}]")
        else:
            print(f"  ⚠️  {label}: {e.response.status_code} {e.response.text[:100]}")
            print(f"      Manual fix: Fabric portal → Workspace → Manage access → Add '{label}' as Contributor")

print("MSI workspace access granted")

In [ ]:
# ============================================================================
# CELL 11 — [Step 6] Create Knowledge Source (OneLake → AI Search index)
# Uses the Foundry IQ Knowledge Sources preview API.
# ============================================================================

print("=" * 60)
print("STEP 6 — Create Knowledge Source (OneLake)")
print("=" * 60)

ks_body = {
    "name": KNOWLEDGE_SOURCE_NAME,
    "description": "Healthcare knowledge files from OneLake",
    "dataSource": {
        "type": "OneLake",
        "workspaceId": FABRIC_WORKSPACE_ID,
        "itemId": FABRIC_LAKEHOUSE_ID,
        "path": KNOWLEDGE_FOLDER_PATH,
    },
    "indexingSettings": {
        "connectionId": SEARCH_CONNECTION_ARM_ID,
        "embeddingModelDeploymentName": EMBEDDING_MODEL,
    }
}

try:
    ks = foundry_request("PUT", f"knowledge/sources/{KNOWLEDGE_SOURCE_NAME}", ks_body)
    if ks is None:
        ks = foundry_request("GET", f"knowledge/sources/{KNOWLEDGE_SOURCE_NAME}")
    KNOWLEDGE_SOURCE_ID = ks.get("id", KNOWLEDGE_SOURCE_NAME)
    KNOWLEDGE_INDEX_NAME = ks.get("indexingSettings", {}).get("indexName", f"{KNOWLEDGE_SOURCE_NAME}-index")
    print(f"  ✅ Knowledge source: {KNOWLEDGE_SOURCE_NAME}")
    print(f"  Index name: {KNOWLEDGE_INDEX_NAME}")
    print(f"  Status: {ks.get('status', 'unknown')} — indexing may take 5-15 min")
except Exception as e:
    print(f"  ⚠️  Knowledge source API not available or returned an error: {e}")
    print("\n  Manual fallback:")
    print("  1. Go to ai.azure.com → your project → Knowledge")
    print("  2. Click '+ Add knowledge source' → OneLake")
    print(f"  3. Workspace: {FABRIC_WORKSPACE_ID}")
    print(f"  4. Lakehouse: {FABRIC_LAKEHOUSE_ID}")
    print(f"  5. Folder path: {KNOWLEDGE_FOLDER_PATH}")
    print(f"  6. Search connection: {SEARCH_CONNECTION_NAME}")
    print(f"  7. Embedding model: {EMBEDDING_MODEL}")
    print("  8. Set KNOWLEDGE_SOURCE_ID and KNOWLEDGE_INDEX_NAME in Cell 12 manually")
    KNOWLEDGE_SOURCE_ID = KNOWLEDGE_SOURCE_NAME
    KNOWLEDGE_INDEX_NAME = f"{KNOWLEDGE_SOURCE_NAME}-index"

In [ ]:
# ============================================================================
# CELL 12 — [Step 7] Create Knowledge Base
# ============================================================================

print("=" * 60)
print("STEP 7 — Create Knowledge Base")
print("=" * 60)

kb_body = {
    "name": KNOWLEDGE_BASE_NAME,
    "displayName": "Healthcare Knowledge Base",
    "description": "Clinical guidelines, formulary, denial management, compliance, and quality measures",
    "modelDeploymentName": CHAT_MODEL,
    "retrievalInstructions": (
        "This knowledge base contains 21 healthcare policy documents for a payer/provider system. "
        "Categories: clinical_guidelines (CHF, COPD, diabetes, preventive care), "
        "denial_management (appeal procedures, denial codes, prevention), "
        "formulary (drug tiers, prior auth, step therapy), "
        "compliance (HIPAA, audit, regulatory reporting), "
        "quality_measures (HEDIS, readmission prevention, provider credentialing). "
        "Always cite document name and section. Prefer specific quantitative thresholds "
        "(e.g. PDC >= 0.80 for adherence, LACE score >= 10 for high readmission risk)."
    ),
    "sources": [{"id": KNOWLEDGE_SOURCE_ID}]
}

try:
    kb = foundry_request("PUT", f"knowledge/bases/{KNOWLEDGE_BASE_NAME}", kb_body)
    if kb is None:
        kb = foundry_request("GET", f"knowledge/bases/{KNOWLEDGE_BASE_NAME}")
    KNOWLEDGE_BASE_ID = kb.get("id", KNOWLEDGE_BASE_NAME)
    print(f"  ✅ Knowledge base: {KNOWLEDGE_BASE_NAME}")
    print(f"  Status: {kb.get('status', 'unknown')}")
except Exception as e:
    print(f"  ⚠️  Knowledge base API error: {e}")
    print("\n  Manual fallback:")
    print("  1. ai.azure.com → Knowledge → Knowledge bases → + Create")
    print(f"  2. Name: {KNOWLEDGE_BASE_NAME}")
    print(f"  3. Model: {CHAT_MODEL}")
    print(f"  4. Source: {KNOWLEDGE_SOURCE_NAME} (from Step 6)")
    print("  5. Add healthcare domain retrieval instructions (see FOUNDRY_IQ_SETUP_GUIDE.md Step 7)")
    KNOWLEDGE_BASE_ID = KNOWLEDGE_BASE_NAME

In [ ]:
# ============================================================================
# CELL 13 — [Step 8] Poll indexer until success (max 20 min)
# If the indexer fails with PermissionDenied, re-check RBAC in Cell 8.
# ============================================================================

print("=" * 60)
print("STEP 8 — Verify Indexer")
print("=" * 60)

INDEXER_NAME = f"{KNOWLEDGE_SOURCE_NAME}-indexer"
MAX_WAIT_SECONDS = 1200  # 20 minutes
POLL_INTERVAL   = 30

headers = {"api-key": SEARCH_ADMIN_KEY, "Content-Type": "application/json"}

start = time.time()
while True:
    elapsed = int(time.time() - start)
    try:
        resp = requests.get(
            f"{SEARCH_ENDPOINT}/indexers/{INDEXER_NAME}/status?api-version=2024-07-01",
            headers=headers, timeout=30
        )
        if resp.status_code == 404:
            print(f"  [{elapsed}s] Indexer not yet created — waiting for Foundry to create it...")
        elif resp.status_code == 200:
            data = resp.json()
            last_result = data.get("lastResult") or {}
            status = last_result.get("status", "unknown")
            docs_ok  = last_result.get("itemsProcessed", 0)
            docs_err = last_result.get("itemsFailed", 0)
            print(f"  [{elapsed}s] Indexer status: {status} | docs succeeded: {docs_ok} | failed: {docs_err}")
            if status == "success" and docs_ok > 0:
                print(f"  ✅ Indexer finished — {docs_ok} documents indexed")
                break
            if status in ("transientFailure", "persistentFailure"):
                errors = last_result.get("errors", [])
                for err in errors[:3]:
                    print(f"     Error: {err.get('errorMessage', err)[:120]}")
                print("  ⚠️  Indexer failed — check RBAC roles (Cell 8) and retry:")
                print(f"     POST {SEARCH_ENDPOINT}/indexers/{INDEXER_NAME}/reset?api-version=2024-07-01")
                print(f"     POST {SEARCH_ENDPOINT}/indexers/{INDEXER_NAME}/run?api-version=2024-07-01")
                break
        else:
            print(f"  [{elapsed}s] HTTP {resp.status_code}: {resp.text[:80]}")
    except Exception as ex:
        print(f"  [{elapsed}s] Error: {ex}")

    if elapsed >= MAX_WAIT_SECONDS:
        print("  ⚠️  Timed out waiting for indexer. Check Azure Portal → AI Search → Indexers.")
        break
    time.sleep(POLL_INTERVAL)

In [ ]:
# ============================================================================
# CELL 14 — [Step 9] Connect Fabric Data Agent to Foundry
#
# ⚠️  KNOWN LIMITATION: The portal UI performs special workspace/artifact
# routing when you pick a Data Agent. The REST API path below attempts
# to replicate this. If the agent tool later returns
# "No tool output found for remote function call", use the manual fallback.
# ============================================================================

print("=" * 60)
print("STEP 9 — Connect Fabric Data Agent → Foundry")
print("=" * 60)

da_connection_body = {
    "name": DATA_AGENT_CONNECTION_NAME,
    "type": "CustomKeys",
    "target": "-",
    "metadata": {
        "type": "fabric_dataagent_preview",
        "workspaceId": FABRIC_WORKSPACE_ID,
        "artifactId": DATA_AGENT_ARTIFACT_ID,
    }
}

try:
    da_conn = foundry_request("PUT", f"connections/{DATA_AGENT_CONNECTION_NAME}", da_connection_body)
    if da_conn is None:
        da_conn = foundry_request("GET", f"connections/{DATA_AGENT_CONNECTION_NAME}")

    DATA_AGENT_CONNECTION_ARM_ID = (
        f"/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP}"
        f"/providers/Microsoft.CognitiveServices/accounts/{HUB_NAME}"
        f"/projects/{PROJECT_NAME}/connections/{DATA_AGENT_CONNECTION_NAME}"
    )
    print(f"  ✅ Data Agent connection: {DATA_AGENT_CONNECTION_NAME}")
    print(f"  Connection ARM ID: {DATA_AGENT_CONNECTION_ARM_ID}")
    print(f"  Verify in Foundry portal: Management center → Connected resources → {DATA_AGENT_CONNECTION_NAME}")
    print(f"  Expected: type=CustomKeys, target='-', metadata.type=fabric_dataagent_preview")

except Exception as e:
    print(f"  ⚠️  Data Agent connection API error: {e}")
    print("\n  Manual fallback (required if above fails):")
    print("  1. ai.azure.com → your project → Management center → Connected resources")
    print("  2. + New connection → Microsoft Fabric → Data Agent")
    print(f"  3. Workspace ID:  {FABRIC_WORKSPACE_ID}")
    print(f"  4. Artifact ID:   {DATA_AGENT_ARTIFACT_ID}")
    print(f"  5. Name: {DATA_AGENT_CONNECTION_NAME}")
    DATA_AGENT_CONNECTION_ARM_ID = (
        f"/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP}"
        f"/providers/Microsoft.CognitiveServices/accounts/{HUB_NAME}"
        f"/projects/{PROJECT_NAME}/connections/{DATA_AGENT_CONNECTION_NAME}"
    )

In [ ]:
# ============================================================================
# CELL 15 — [Step 10] Create Orchestrator Agent
#
# CRITICAL: azure_ai_search is added as GROUNDING in the tools list,
# NOT as an MCP server. Adding it as MCP causes a server_authentication
# error that breaks the API (known Foundry bug).
# ============================================================================

print("=" * 60)
print("STEP 10 — Create Orchestrator Agent")
print("=" * 60)

# Load instructions from the repo
instructions_path = Path("foundry_agent/orchestrator_instructions.md")
if instructions_path.exists():
    raw = instructions_path.read_text(encoding="utf-8")
    # Strip YAML front matter (first 4 comment lines)
    lines = raw.splitlines()
    start = next((i for i, l in enumerate(lines) if l.strip() == "---" and i > 0), 4) + 1
    AGENT_INSTRUCTIONS = "\n".join(lines[start:]).strip()
    print(f"  Loaded instructions: {len(AGENT_INSTRUCTIONS):,} chars from {instructions_path}")
else:
    raise FileNotFoundError(
        f"Instructions not found at {instructions_path}. "
        "Clone the full repo or adjust the path."
    )

agent_body = {
    "name": ORCHESTRATOR_AGENT_NAME,
    "displayName": "Healthcare Orchestrator (Grounded KB)",
    "description": (
        "Healthcare orchestrator with Knowledge Base grounding (clinical guidelines), "
        "Fabric Data Agent (structured lakehouse data), and web search."
    ),
    "definition": {
        "kind": "prompt",
        "model": CHAT_MODEL,
        "instructions": AGENT_INSTRUCTIONS,
        "tools": [
            # Tool 1 — Web search
            {"type": "web_search_preview"},
            # Tool 2 — Fabric Data Agent (structured data)
            {
                "type": "fabric_dataagent_preview",
                "fabric_dataagent_preview": {
                    "project_connections": [
                        {"project_connection_id": DATA_AGENT_CONNECTION_ARM_ID}
                    ]
                }
            },
            # Tool 3 — Azure AI Search GROUNDING (NOT MCP — see note above)
            {
                "type": "azure_ai_search",
                "azure_ai_search": {
                    "indexes": [
                        {
                            "project_connection_id": SEARCH_CONNECTION_ARM_ID,
                            "index_name": KNOWLEDGE_INDEX_NAME
                        }
                    ]
                }
            }
        ]
    }
}

# Create or update
agent_result = foundry_request("POST", "agents", agent_body, api_version="v1")
if agent_result is None:
    # Try PUT to overwrite existing
    agent_result = foundry_request(
        "PUT", f"agents/{ORCHESTRATOR_AGENT_NAME}", agent_body, api_version="v1"
    )

AGENT_ID = agent_result.get("id", ORCHESTRATOR_AGENT_NAME)
print(f"  ✅ Agent created: {ORCHESTRATOR_AGENT_NAME}")
print(f"  Agent ID: {AGENT_ID}")
print(f"  Model:    {CHAT_MODEL}")
print(f"  Tools:    web_search_preview, fabric_dataagent_preview, azure_ai_search")
print(f"  View in Foundry portal: ai.azure.com → {PROJECT_NAME} → Agents")

In [ ]:
# ============================================================================
# CELL 16 — [Step 11] Validate: run 3 test queries against the agent
#
# Tests: (1) knowledge-only, (2) data-only, (3) combined
# ============================================================================

print("=" * 60)
print("STEP 11 — Validate Agent")
print("=" * 60)

TEST_QUERIES = [
    {
        "label": "Knowledge-only (KB grounding)",
        "query": "What does our CHF management guideline recommend for readmission prevention?",
        "expect": "KB document citation",
    },
    {
        "label": "Data-only (Fabric Data Agent)",
        "query": "Show me denial rates by payer",
        "expect": "table with payer names and denial rates",
    },
    {
        "label": "Combined (both tools)",
        "query": "What is our denial rate by payer and what does our appeal process guide recommend for the top denial reasons?",
        "expect": "denial rate table + KB citation for appeal procedures",
    },
]


def run_agent_query(query: str) -> str:
    """Create a thread, post a message, poll for completion, return final text."""
    # Create thread
    thread = foundry_request("POST", f"agents/{AGENT_ID}/threads", {}, api_version="v1")
    thread_id = thread["id"]

    # Post user message
    foundry_request(
        "POST", f"agents/{AGENT_ID}/threads/{thread_id}/messages",
        {"role": "user", "content": query}, api_version="v1"
    )

    # Create and poll run
    run = foundry_request(
        "POST", f"agents/{AGENT_ID}/threads/{thread_id}/runs",
        {"assistant_id": AGENT_ID}, api_version="v1"
    )
    run_id = run["id"]

    for _ in range(60):  # max 5 min
        time.sleep(5)
        run_status = foundry_request(
            "GET", f"agents/{AGENT_ID}/threads/{thread_id}/runs/{run_id}",
            api_version="v1"
        )
        status = run_status.get("status", "")
        if status == "completed":
            break
        if status in ("failed", "cancelled", "expired"):
            return f"[Run {status}] {run_status.get('last_error', {}).get('message', '')}"
        print(f"    ... {status}", end="\r")

    # Retrieve messages
    msgs = foundry_request(
        "GET", f"agents/{AGENT_ID}/threads/{thread_id}/messages",
        api_version="v1"
    )
    assistant_msgs = [
        m for m in (msgs.get("data") or msgs.get("value", []))
        if m.get("role") == "assistant"
    ]
    if not assistant_msgs:
        return "[No assistant response]"

    content = assistant_msgs[-1].get("content", [])
    texts = [c["text"]["value"] for c in content if c.get("type") == "text"]
    return "\n".join(texts)


for i, test in enumerate(TEST_QUERIES, 1):
    print(f"\nTest {i}/3 — {test['label']}")
    print(f"  Query:  {test['query']}")
    print(f"  Expect: {test['expect']}")
    try:
        answer = run_agent_query(test["query"])
        preview = answer[:400].replace("\n", " ")
        print(f"  Answer: {preview}...")
        print(f"  ✅ Test {i} passed")
    except Exception as ex:
        print(f"  ⚠️  Test {i} error: {ex}")
        print(f"     See FOUNDRY_ORCHESTRATOR_TROUBLESHOOTING.md for diagnosis steps.")

print("\n" + "=" * 60)
print("DEPLOYMENT COMPLETE")
print("=" * 60)
print(f"  Foundry portal: https://ai.azure.com")
print(f"  Project:        {PROJECT_NAME}")
print(f"  Agent:          {ORCHESTRATOR_AGENT_NAME}")
print(f"  KB:             {KNOWLEDGE_BASE_NAME} ({KNOWLEDGE_INDEX_NAME})")
print(f"  Fabric agent:   {DATA_AGENT_CONNECTION_NAME}")

## Troubleshooting Quick Reference

| Symptom | Cause | Fix |
|---------|-------|-----|
| Indexer `PermissionDenied` | Search MSI missing RBAC | Re-run Cell 8, wait 10 min, reset+run indexer |
| `No tool output found` from Fabric agent | Data Agent connection has empty workspace routing | Delete connection in portal, manually recreate (Cell 14 manual fallback) |
| Agent returns `server_authentication` error | KB added as MCP tool instead of grounding | Delete agent, re-run Cell 15 (uses `azure_ai_search` tool type, not MCP) |
| Agent only answers KB, ignores data | Orchestrator instructions stripped or truncated | Verify instructions length > 5000 chars; Cell 15 loads from `foundry_agent/orchestrator_instructions.md` |
| Data agent returns wrong aggregations | Fewshot phrasing mismatch | Use EXACT phrases from the Query Catalog in `orchestrator_instructions.md` |
| Cells 11-12 raise 404 / 405 | Knowledge Source/Base APIs still preview | Use manual portal fallback steps printed in each cell |

See [FOUNDRY_ORCHESTRATOR_TROUBLESHOOTING.md](FOUNDRY_ORCHESTRATOR_TROUBLESHOOTING.md) for detailed diagnostic workflows.

---

## Post-Run Manual Checklist

Complete these steps **after** the notebook finishes. Check each one before testing the agent.

---

### 🔴 Step 1 — Fabric Data Agent Connection (do this first)

> Required if Cell 14 printed a warning. Even if Cell 14 succeeded, verify the connection has correct workspace routing before proceeding.

1. Go to [ai.azure.com](https://ai.azure.com) → your project → **Management center** (bottom-left) → **Connected resources**
2. If a connection named `HealthcareHLSAgent` already exists, click it and confirm the details:
   - `type` = `CustomKeys`
   - `target` = `-`
   - `metadata.type` = `fabric_dataagent_preview`
   - If it looks empty or wrong → **delete it**, then continue below
3. Click **+ New connection → Microsoft Fabric → Data Agent**
4. Enter:
   - **Workspace ID**: *(value from Cell 2 `FABRIC_WORKSPACE_ID`)*
   - **Artifact ID**: *(value from Cell 2 `DATA_AGENT_ARTIFACT_ID`)*
   - **Connection name**: `HealthcareHLSAgent`
5. Click **Connect**

> To find Workspace ID and Artifact ID: Fabric portal → open `HealthcareHLSAgent` → copy from the browser URL:
> `https://app.fabric.microsoft.com/groups/<WORKSPACE_ID>/aiskills/<ARTIFACT_ID>`

---

### 🟡 Step 2 — Knowledge Source (only if Cell 11 printed a warning)

1. Go to [ai.azure.com](https://ai.azure.com) → your project → **Knowledge**
2. Click **+ Add knowledge source → OneLake**
3. Configure:
   - **Workspace**: select your Fabric workspace
   - **Lakehouse**: select `lh_bronze_raw` (or the lakehouse ID from Cell 2)
   - **Folder path**: `Files/healthcare_knowledge`
   - **Search connection**: `healthcareaisearch` *(created by Cell 9)*
   - **Embedding model**: `text-embedding-ada-002`
4. Click **Create** — status will show *Indexing* for 5–15 minutes, then *Active*

---

### 🟡 Step 3 — Knowledge Base (only if Cell 12 printed a warning)

1. Go to [ai.azure.com](https://ai.azure.com) → your project → **Knowledge → Knowledge bases**
2. Click **+ Create knowledge base**
3. Configure:
   - **Name**: `healthcareknowledgebase`
   - **Model**: `gpt-4o`
   - **Source**: select the knowledge source created in Step 2
4. Add retrieval instructions:
   > *This knowledge base contains 21 healthcare policy documents for a payer/provider system covering clinical guidelines (CHF, COPD, diabetes, preventive care), denial management, formulary, compliance, and quality measures. Always cite document name and section.*
5. Click **Create**

---

### 🟡 Step 4 — Set `KNOWLEDGE_INDEX_NAME` and re-run Cell 15 (only if Steps 2–3 were manual)

The agent creation in Cell 15 needs the exact AI Search index name that Foundry generated.

1. Go to **Azure Portal → AI Search service (`healthcarefoundryais`) → Indexes**
2. Copy the index name (e.g. `healthcare-onelake-ks-index`)
3. In Cell 15, update the line:
   ```python
   KNOWLEDGE_INDEX_NAME = "paste-your-index-name-here"
   ```
4. Re-run Cell 15 to recreate the agent with the correct index wired in

---

### ✅ Step 5 — Verify the Agent in Foundry Playground

Run these three queries in the **Foundry Playground** (`ai.azure.com → Agents → HealthcareOrchestratorAgent → Test`):

| # | Query | What to check |
|---|-------|---------------|
| 1 | `What does our CHF management guideline recommend for readmission prevention?` | Response cites a KB document by name (e.g. `CHF_Management_Guidelines.md`) |
| 2 | `Show me denial rates by payer` | Returns a table with 12 payers and numeric denial rates |
| 3 | `What is our denial rate by payer and what does our appeal process guide recommend for the top denial reasons?` | Both a data table AND a KB document citation appear in the same response |

If query 2 returns *"No tool output found"* → redo Manual Step 1.
If query 1 returns no document citation → redo Manual Steps 2–4.
If either returns a `server_authentication` error → the KB was accidentally added as an MCP tool; re-run Cell 15 (it uses `azure_ai_search` grounding, not MCP).

See [FOUNDRY_ORCHESTRATOR_TROUBLESHOOTING.md](FOUNDRY_ORCHESTRATOR_TROUBLESHOOTING.md) for detailed diagnosis.
